In [1]:
!pip install pytorch-forecasting lightning lightgbm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.8/399.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 63.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
import lightning.pytorch as pl
import lightgbm as lgb

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE

from lightning.pytorch.callbacks import EarlyStopping

In [5]:
df_bank = pd.read_csv("/content/combinedBankData.csv")
df_tech = pd.read_csv("/content/technoFundamentalData.csv")

# Strip whitespace from all column names
df_bank.columns = df_bank.columns.str.strip()
df_tech.columns = df_tech.columns.str.strip()

# Rename 'symbol' to 'bank' in df_tech to facilitate merging if necessary
if 'symbol' in df_tech.columns and 'bank' not in df_tech.columns:
    df_tech = df_tech.rename(columns={'symbol': 'bank'})

# Merge the dataframes
df = df_bank.merge(df_tech, on=["date", "bank"], how="left")

# Convert date to datetime and sort
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["bank", "date"])
df.head()

,date,open,high,low,close_x,per_change,volume_x,amount,bank,nepse_close,...,volume_y,time_idx,log_return,momentum_20d,volatility_20d,volume_z,target_close,day_of_week,is_gap,risk_stability
0,2011-11-13,119.0,120.0,118.0,120.0,0.84,1164.0,139094.0,ADBL,331.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2011-11-14,120.0,122.0,114.0,122.0,1.67,2691.0,314662.0,ADBL,331.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2011-11-15,121.0,121.0,118.0,119.0,-2.46,1157.0,138490.0,ADBL,327.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2011-11-16,119.0,120.0,118.0,120.0,0.84,1400.0,166860.0,ADBL,326.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2011-11-17,120.0,120.0,118.0,118.0,-1.67,734.0,86866.0,ADBL,329.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Use 'close_x' as the target closing price since 'close' was renamed during merge
# Group by bank to ensure shifts don't cross between different banks
df["t1"] = df.groupby("bank")["close_x"].shift(-1)
df["t1"] = (df["t1"] - df["close_x"]) / df["close_x"]

df["t3"] = df.groupby("bank")["close_x"].shift(-3)
df["t3"] = (df["t3"] - df["close_x"]) / df["close_x"]

df["t5"] = df.groupby("bank")["close_x"].shift(-5)
df["t5"] = (df["t5"] - df["close_x"]) / df["close_x"]

# Drop rows where targets couldn't be calculated (end of sequences)
df = df.dropna(subset=["t1", "t3", "t5"])

# Create a proper time_idx for pytorch-forecasting
df["time_idx"] = df.groupby("bank").cumcount()

df.head()

/tmp/ipykernel_822/73635065.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["time_idx"] = df.groupby("bank").cumcount()


,date,open,high,low,close_x,per_change,volume_x,amount,bank,nepse_close,...,momentum_20d,volatility_20d,volume_z,target_close,day_of_week,is_gap,risk_stability,t1,t3,t5
0,2011-11-13,119.0,120.0,118.0,120.0,0.84,1164.0,139094.0,ADBL,331.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.016667,0.000000,-0.016667
1,2011-11-14,120.0,122.0,114.0,122.0,1.67,2691.0,314662.0,ADBL,331.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.024590,-0.032787,-0.032787
2,2011-11-15,121.0,121.0,118.0,119.0,-2.46,1157.0,138490.0,ADBL,327.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008403,-0.008403,-0.016807
3,2011-11-16,119.0,120.0,118.0,120.0,0.84,1400.0,166860.0,ADBL,326.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.016667,-0.016667,-0.025000
4,2011-11-17,120.0,120.0,118.0,118.0,-1.67,734.0,86866.0,ADBL,329.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,-0.008475,-0.008475


In [8]:
max_time = df["time_idx"].max()
split = int(max_time * 0.85)

train_df = df[df.time_idx <= split]
val_df   = df[df.time_idx > split]

In [10]:
max_encoder_length = 30
max_prediction_length = 5

training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="t5",  # main horizon
    group_ids=["bank"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    # Use the correct column names from the merged dataframe
    time_varying_unknown_reals=["close_x", "volume_x", "t1", "t3", "t5"],
    time_varying_known_reals=["time_idx"],

    target_normalizer=GroupNormalizer(groups=["bank"]),
)

validation = TimeSeriesDataSet.from_dataset(training, val_df, stop_randomization=True)

train_loader = training.to_dataloader(train=True, batch_size=64)
val_loader   = validation.to_dataloader(train=False, batch_size=64)

In [11]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.2,
    hidden_continuous_size=16,
    loss=MAE(),
)

# Configure EarlyStopping with verbose=True to see logs when it triggers
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    verbose=True,
    mode="min"
)

trainer = pl.Trainer(
    max_epochs=15,
    accelerator="auto",
    enable_model_summary=True,
    log_every_n_steps=10,
    callbacks=[early_stop_callback],
    enable_checkpointing=True,
    enable_progress_bar=True
)

trainer.fit(tft, train_loader, val_loader)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MAE                             │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      0 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    192 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │      0 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 11.9 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  1.8 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     33 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 66.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 66.5 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 236                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: Metric val_loss improved. New best score: 0.022
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved. New best score: 0.022
INFO: Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.022
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.022
INFO: Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.022
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.022
INFO: Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.022
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.022
INFO: Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.022
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.022
INFO: Metric val_loss impr

In [16]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

preds = tft.predict(val_loader).flatten()
actuals = torch.cat([y[0] for x, y in iter(val_loader)]).flatten()

# Regression Metric
mae = torch.mean(torch.abs(preds - actuals)).item()
print("\n=== TFT VALIDATION ===")
print(f"MAE: {mae:.5f}")

# Classification Metrics (Directional Accuracy)
# Convert returns to binary (1 if positive, 0 if negative/flat)
y_true_bin = (actuals > 0).numpy().astype(int)
y_pred_score = preds.numpy()  # Use raw prediction as a probability proxy for AUC
y_pred_bin = (preds > 0).numpy().astype(int)

print("\n=== DIRECTIONAL METRICS ===")
print(f"ROC AUC: {roc_auc_score(y_true_bin, y_pred_score):.4f}")
print(f"Accuracy: {accuracy_score(y_true_bin, y_pred_bin):.4f}")
print(f"F1 Score: {f1_score(y_true_bin, y_pred_bin):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_bin, y_pred_bin))

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch


=== TFT VALIDATION ===
MAE: 0.02143

=== DIRECTIONAL METRICS ===
ROC AUC: 0.7939
Accuracy: 0.7246
F1 Score: 0.6200

Confusion Matrix:
[[10865  1174]
 [ 4812  4884]]


In [13]:
features = [col for col in df.columns if col not in ["date","bank","t1","t3","t5"]]

X_train = train_df[features]
y_train = train_df["t5"]

X_val = val_df[features]
y_val = val_df["t5"]

lgb_model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6
)

lgb_model.fit(X_train, y_train)

lgb_preds = lgb_model.predict(X_val)

print("\n=== LIGHTGBM VALIDATION ===")
print("MAE:", np.mean(np.abs(lgb_preds - y_val)))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002285 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3408
[LightGBM] [Info] Number of data points in the train set: 45199, number of used features: 16
[LightGBM] [Info] Start training from score 0.001090
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [14]:
tft_np = preds.detach().cpu().numpy()

min_len = min(len(tft_np), len(lgb_preds))
ensemble = 0.5 * tft_np[:min_len] + 0.5 * lgb_preds[:min_len]

print("\n=== ENSEMBLE ===")
print("MAE:", np.mean(np.abs(ensemble - y_val.values[:min_len])))


=== ENSEMBLE ===
MAE: 0.028692692380450815


In [15]:
print("\n=== WALK-FORWARD CV ===")

folds = 3
fold_size = int(len(df) * 0.1)

scores = []

for i in range(folds):
    cutoff = split + i * fold_size
    train_fold = df[df.time_idx <= cutoff]
    val_fold   = df[(df.time_idx > cutoff) & (df.time_idx <= cutoff + fold_size)]

    if len(val_fold) == 0:
        continue

    X_tr = train_fold[features]
    y_tr = train_fold["t5"]

    X_va = val_fold[features]
    y_va = val_fold["t5"]

    model = lgb.LGBMRegressor(n_estimators=200)
    model.fit(X_tr, y_tr)

    preds = model.predict(X_va)
    score = np.mean(np.abs(preds - y_va))

    scores.append(score)

print("CV MAE:", np.mean(scores))


=== WALK-FORWARD CV ===
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007470 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3408
[LightGBM] [Info] Number of data points in the train set: 45199, number of used features: 16
[LightGBM] [Info] Start training from score 0.001090
CV MAE: 0.024623580682033432


In [18]:
full_dataset = TimeSeriesDataSet.from_dataset(training, df)
full_loader = full_dataset.to_dataloader(train=True, batch_size=64)

# Re-initialize model with the same hyperparameters
tft_full = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.2,
    hidden_continuous_size=16,
    loss=MAE(),
)

# Create a NEW trainer instance for the full training run
trainer_full = pl.Trainer(
    max_epochs=15,
    accelerator="auto",
    enable_model_summary=True,
    log_every_n_steps=10,
    enable_checkpointing=True
)

trainer_full.fit(tft_full, full_loader)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MAE                             │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      0 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    192 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │      0 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 11.9 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  1.8 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     33 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 66.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 66.5 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 236                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/training_epoch_loop.py:500: ReduceLROnPlateau 
conditioned on metric val_loss which is not available but strict is set to `False`. Skipping learning rate update.

INFO: `Trainer.fit` stopped: `max_epochs=15` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=15` reached.


In [19]:
print("\n=== LIGHTGBM FEATURE IMPORTANCE ===")

imp = pd.Series(lgb_model.feature_importances_, index=features)
print(imp.sort_values(ascending=False).head(10))



=== LIGHTGBM FEATURE IMPORTANCE ===
nepse_close      1874
nepse_ret_21d    1468
nepse_ret_5d     1234
nepse_ret_1d      785
time_idx          696
per_change        409
amount            406
npl               320
car               314
volume_x          249
dtype: int32


In [20]:
torch.save(tft_full.state_dict(), "tft_model.pth")